In [110]:
# 1. Importación de librerías
from pathlib import Path
import polars as pl

# 2. Definir la ruta del dataset
RUTA_PROYECTO = Path.cwd().parent
RUTA_DATASET = RUTA_PROYECTO / "data" / "dataset_proyecto.csv"

# 3. Cargar los datos
df = pl.read_csv(RUTA_DATASET)

# 4. Verificar la carga
print(f"Dimensiones del dataset: {df.shape}")
print(f"Ruta del dataset: {RUTA_DATASET}")
print("\nPrimeras 5 filas:")
print(df.head())

Dimensiones del dataset: (6362620, 11)
Ruta del dataset: d:\ML_III\data\dataset_proyecto.csv

Primeras 5 filas:
shape: (5, 11)
┌──────┬──────────┬──────────┬─────────────┬───┬─────────────┬─────────────┬─────────┬─────────────┐
│ step ┆ type     ┆ amount   ┆ nameOrig    ┆ … ┆ oldbalanceD ┆ newbalanceD ┆ isFraud ┆ isFlaggedFr │
│ ---  ┆ ---      ┆ ---      ┆ ---         ┆   ┆ est         ┆ est         ┆ ---     ┆ aud         │
│ i64  ┆ str      ┆ f64      ┆ str         ┆   ┆ ---         ┆ ---         ┆ i64     ┆ ---         │
│      ┆          ┆          ┆             ┆   ┆ f64         ┆ f64         ┆         ┆ i64         │
╞══════╪══════════╪══════════╪═════════════╪═══╪═════════════╪═════════════╪═════════╪═════════════╡
│ 1    ┆ PAYMENT  ┆ 9839.64  ┆ C1231006815 ┆ … ┆ 0.0         ┆ 0.0         ┆ 0       ┆ 0           │
│ 1    ┆ PAYMENT  ┆ 1864.28  ┆ C1666544295 ┆ … ┆ 0.0         ┆ 0.0         ┆ 0       ┆ 0           │
│ 1    ┆ TRANSFER ┆ 181.0    ┆ C1305486145 ┆ … ┆ 0.0         ┆ 0.

In [111]:
# 5. Revisar la estructura del dataset
print("Columnas y tipos de datos:")
print(df.schema)

print("\nCantidad de valores nulos por columna:")
print(df.null_count())

print("\nNúmero de filas duplicadas:")
print(df.is_duplicated().sum())

Columnas y tipos de datos:
Schema([('step', Int64), ('type', String), ('amount', Float64), ('nameOrig', String), ('oldbalanceOrg', Float64), ('newbalanceOrig', Float64), ('nameDest', String), ('oldbalanceDest', Float64), ('newbalanceDest', Float64), ('isFraud', Int64), ('isFlaggedFraud', Int64)])

Cantidad de valores nulos por columna:
shape: (1, 11)
┌──────┬──────┬────────┬──────────┬───┬────────────────┬────────────────┬─────────┬────────────────┐
│ step ┆ type ┆ amount ┆ nameOrig ┆ … ┆ oldbalanceDest ┆ newbalanceDest ┆ isFraud ┆ isFlaggedFraud │
│ ---  ┆ ---  ┆ ---    ┆ ---      ┆   ┆ ---            ┆ ---            ┆ ---     ┆ ---            │
│ u32  ┆ u32  ┆ u32    ┆ u32      ┆   ┆ u32            ┆ u32            ┆ u32     ┆ u32            │
╞══════╪══════╪════════╪══════════╪═══╪════════════════╪════════════════╪═════════╪════════════════╡
│ 0    ┆ 0    ┆ 0      ┆ 0        ┆ … ┆ 0              ┆ 0              ┆ 0       ┆ 0              │
└──────┴──────┴────────┴──────────┴───┴──

In [112]:
# 6. Distribución de la variable objetivo
distribucion = (
    df.group_by("isFraud")
    .len()
    .sort("isFraud")
    .with_columns(
        (pl.col("len") / df.height * 100).alias("porcentaje")
    )
)

print(distribucion)

shape: (2, 3)
┌─────────┬─────────┬────────────┐
│ isFraud ┆ len     ┆ porcentaje │
│ ---     ┆ ---     ┆ ---        │
│ i64     ┆ u32     ┆ f64        │
╞═════════╪═════════╪════════════╡
│ 0       ┆ 6354407 ┆ 99.870918  │
│ 1       ┆ 8213    ┆ 0.129082   │
└─────────┴─────────┴────────────┘


In [113]:
# 7. División temporal de los datos
df_train = df.filter(pl.col("step") <= 400)
df_prod = df.filter(pl.col("step") >= 401)

print("Conjunto de entrenamiento:", df_train.shape)
print("Conjunto de producción:", df_prod.shape)

print("\nPeriodo de entrenamiento:")
print(df_train.select(
    pl.col("step").min().alias("step_min"),
    pl.col("step").max().alias("step_max")
))

print("\nPeriodo de producción:")
print(df_prod.select(
    pl.col("step").min().alias("step_min"),
    pl.col("step").max().alias("step_max")
))

Conjunto de entrenamiento: (5787030, 11)
Conjunto de producción: (575590, 11)

Periodo de entrenamiento:
shape: (1, 2)
┌──────────┬──────────┐
│ step_min ┆ step_max │
│ ---      ┆ ---      │
│ i64      ┆ i64      │
╞══════════╪══════════╡
│ 1        ┆ 400      │
└──────────┴──────────┘

Periodo de producción:
shape: (1, 2)
┌──────────┬──────────┐
│ step_min ┆ step_max │
│ ---      ┆ ---      │
│ i64      ┆ i64      │
╞══════════╪══════════╡
│ 401      ┆ 743      │
└──────────┴──────────┘


In [114]:
# 8. Definir la variable objetivo y los predictores
TARGET = "isFraud"

FEATURES = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

# Separar predictores y variable objetivo
X_train = df_train.select(FEATURES)
y_train = df_train[TARGET]

X_prod = df_prod.select(FEATURES)
y_prod = df_prod[TARGET]

print("Variables predictoras:", FEATURES)
print("Variable objetivo:", TARGET)

print("\nDimensiones de entrenamiento:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nDimensiones de producción:")
print("X_prod:", X_prod.shape)
print("y_prod:", y_prod.shape)

Variables predictoras: ['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
Variable objetivo: isFraud

Dimensiones de entrenamiento:
X_train: (5787030, 7)
y_train: (5787030,)

Dimensiones de producción:
X_prod: (575590, 7)
y_prod: (575590,)


In [115]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Identificar variables numéricas
NUMERIC_FEATURES = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

# Configurar el preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["type"])
    ],
    sparse_threshold=1.0
)

print("Preprocesador configurado correctamente.")

Preprocesador configurado correctamente.


In [116]:
# Ajustar únicamente con los datos de entrenamiento
preprocessor.fit(X_train)

# Crear carpeta para guardar el preprocesador
from pathlib import Path
import joblib

RUTA_MODELOS = Path("../models")
RUTA_MODELOS.mkdir(parents=True, exist_ok=True)

# Guardar el preprocesador ajustado
joblib.dump(
    preprocessor,
    RUTA_MODELOS / "preprocessor.joblib"
)

print("Preprocesador ajustado y guardado correctamente.")
print("Número de variables transformadas:",
      len(preprocessor.get_feature_names_out()))
print("Categorías de 'type':",
      preprocessor.named_transformers_["cat"].categories_[0])

Preprocesador ajustado y guardado correctamente.
Número de variables transformadas: 11
Categorías de 'type': ['CASH_IN' 'CASH_OUT' 'DEBIT' 'PAYMENT' 'TRANSFER']


In [118]:
# Validar la transformación con una muestra pequeña
X_muestra = X_train.head(1000).to_dict(as_series=False)
X_muestra = pl.DataFrame(X_muestra)

# Convertir el diccionario a Pandas directamente
import pandas as pd
X_muestra = pd.DataFrame(X_muestra.to_dict(as_series=False))

# Aplicar el preprocesador
X_transformado = preprocessor.transform(X_muestra)

print("Dimensiones originales:", X_muestra.shape)
print("Dimensiones transformadas:", X_transformado.shape)
print("Preprocesamiento validado:", X_transformado.shape[1] == 11)

Dimensiones originales: (1000, 7)
Dimensiones transformadas: (1000, 11)
Preprocesamiento validado: True
